In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/unnormalized_orders.csv")

In [4]:
print("=== SHAPE ===")
print(f"{df.shape[0]} rows, {df.shape[1]} columns\n")

print("=== DTYPES ===")
print(df.dtypes, "\n")

print("=== FIRST 5 ROWS ===")
print(df.head(), "\n")

=== SHAPE ===
310 rows, 9 columns

=== DTYPES ===
Order_id             int64
date                object
customer_id          int64
customer_name       object
product_id         float64
product_name        object
Price(Rs)          float64
quantity           float64
total_price(Rs)    float64
dtype: object 

=== FIRST 5 ROWS ===
   Order_id        date  customer_id      customer_name  product_id  \
0      1236  01/01/2024           74    Arjun, 40, Male       148.0   
1      1428  01/01/2024           54  Priya, 33, Female       149.0   
2      1585  01/01/2024           71     Liam, 42, Male       248.0   
3      1240  02/01/2024           56   Emma, 53, Female       242.0   
4      1311  03/01/2024           79    Ayaan, 32, Male       299.0   

      product_name  Price(Rs)  quantity  total_price(Rs)  
0        1. Apples      150.0      13.0           1950.0  
1       2. Bananas       40.0       1.0             40.0  
2       3. Oranges       80.0       3.0            240.0  
3      

In [5]:
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
print(missing_df[missing_df["missing_count"] > 0] if missing.any() else "No missing values found", "\n")

=== MISSING VALUES ===
                 missing_count  missing_%
product_name                10       3.23
Price(Rs)                   10       3.23
quantity                    15       4.84
total_price(Rs)             25       8.06 



In [6]:
print("=== DESCRIPTIVE STATS (numeric columns) ===")
print(df.describe().round(2), "\n")

=== DESCRIPTIVE STATS (numeric columns) ===
       Order_id  customer_id  product_id  Price(Rs)  quantity  total_price(Rs)
count    310.00       310.00      310.00     300.00    295.00           285.00
mean    1359.44        54.29      205.09     282.20      9.11          2526.11
std      143.83        26.08       52.83     262.51      6.11          3217.10
min     1111.00        10.00      116.00      20.00      1.00            40.00
25%     1231.25        32.00      158.75      60.00      3.00           400.00
50%     1366.50        55.00      201.00     200.00      9.00          1200.00
75%     1482.50        74.00      252.75     400.00     15.00          3200.00
max     1599.00       100.00      299.00    1000.00     19.00         17000.00 



In [7]:
print("=== OUTLIERS (IQR method) ===")
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"[{col}] bounds: ({lower:.2f}, {upper:.2f}) → {len(outliers)} outlier(s)")
    if not outliers.empty:
        print(outliers[["Order_id", col]], "\n")

=== OUTLIERS (IQR method) ===
[Order_id] bounds: (854.38, 1859.38) → 0 outlier(s)
[customer_id] bounds: (-31.00, 137.00) → 0 outlier(s)
[product_id] bounds: (17.75, 393.75) → 0 outlier(s)
[Price(Rs)] bounds: (-450.00, 910.00) → 6 outlier(s)
     Order_id  Price(Rs)
47       1498     1000.0
97       1539     1000.0
147      1448     1000.0
197      1275     1000.0
247      1544     1000.0
297      1324     1000.0 

[quantity] bounds: (-15.00, 33.00) → 0 outlier(s)
[total_price(Rs)] bounds: (-3800.00, 7400.00) → 26 outlier(s)
     Order_id  total_price(Rs)
5        1406          15200.0
24       1405          13600.0
31       1521          10400.0
74       1526           9600.0
87       1546           8000.0
91       1166          10400.0
97       1539          12000.0
99       1510          11400.0
105      1208          10400.0
123      1298           7600.0
138      1134          10500.0
143      1329           7600.0
149      1517          10800.0
174      1142          12800.0
180  

In [8]:
print("\n=== ANOMALIES ===")

# Duplicate order IDs
dupes = df[df.duplicated(subset="Order_id", keep=False)]
print(f"Duplicate Order_ids: {len(dupes)}")
if not dupes.empty:
    print(dupes[["Order_id", "customer_name", "product_name"]], "\n")

# Negative or zero quantity / price
for col in ["quantity", "Price(Rs)", "total_price(Rs)"]:
    bad = df[df[col] <= 0]
    print(f"[{col}] zero or negative values: {len(bad)}")
    if not bad.empty:
        print(bad[["Order_id", col]], "\n")

# Orders with no matching customer (nulls after merge)
no_customer = df[df["customer_name"].isnull()]
print(f"\nOrders with no matching customer: {len(no_customer)}")

# Orders with no matching product
no_product = df[df["product_name"].isnull()]
print(f"Orders with no matching product: {len(no_product)}")

# total_price mismatch (sanity check: quantity * price should equal total)
df["_expected_total"] = df["quantity"] * df["Price(Rs)"]
mismatch = df[~np.isclose(df["total_price(Rs)"], df["_expected_total"], equal_nan=True)]
print(f"\ntotal_price(Rs) mismatches: {len(mismatch)}")
if not mismatch.empty:
    print(mismatch[["Order_id", "quantity", "Price(Rs)", "total_price(Rs)", "_expected_total"]])


=== ANOMALIES ===
Duplicate Order_ids: 6
     Order_id      customer_name   product_name
6        1457     Noah, 53, Male  7. Pineapples
12       1457     Noah, 53, Male     13. Onions
51       1162  William, 19, Male     2. Bananas
52       1162  William, 19, Male     3. Oranges
107      1563    Arjun, 40, Male     8. Mangoes
112      1563    Arjun, 40, Male     13. Onions 

[quantity] zero or negative values: 0
[Price(Rs)] zero or negative values: 0
[total_price(Rs)] zero or negative values: 0

Orders with no matching customer: 0
Orders with no matching product: 10

total_price(Rs) mismatches: 0


In [9]:
print("\n=== VALUE COUNTS ===")
for col in ["product_name"]:
    print(f"\n[{col}]\n", df[col].value_counts())


=== VALUE COUNTS ===

[product_name]
 product_name
1. Apples                6
38. Beef                 6
28. Flour                6
29. Sugar                6
30. Salt                 6
31. Olive oil            6
32. Coffee               6
33. Tea                  6
34. Cereal               6
35. Peanut butter        6
36. Jelly                6
37. Chicken              6
39. Fish                 6
2. Bananas               6
40. Pork                 6
41. Shampoo              6
42. Toothpaste           6
43. Soap                 6
44. Laundry detergent    6
45. Paper towels         6
46. Toilet paper         6
47. Light bulbs          6
48. Batteries            6
49. Dish soap            6
27. Pasta                6
26. Rice                 6
25. Cheese               6
24. Butter               6
3. Oranges               6
4. Grapes                6
5. Strawberries          6
6. Blueberries           6
7. Pineapples            6
8. Mangoes               6
9. Watermelons           6
10.